In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install libraries  [Run this first]
# ════════════════════════════════════════════════════════════════════════════

# pandas and numpy are already installed in Colab — this just confirms versions
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

print("✓ pandas  version:", pd.__version__)
print("✓ numpy   version:", np.__version__)
print("✓ Libraries ready — run Cell 2 to upload your CSV file")

✓ pandas  version: 2.2.2
✓ numpy   version: 2.0.2
✓ Libraries ready — run Cell 2 to upload your CSV file


In [2]:
# CELL 2 — Upload the CSV file  [Run this after Cell 1]
# ════════════════════════════════════════════════════════════════════════════

from google.colab import files

print("A file chooser will appear below.")
print("Select: survey_results_public.csv (the raw Stack Overflow dataset)")
print("File size is ~135 MB — upload may take 1-2 minutes on slower connections.")
print()

uploaded = files.upload()   # opens file chooser dialog

# Get the filename that was uploaded
filename = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {filename}")
print(f"✓ Size: {len(uploaded[filename]) / 1024 / 1024:.1f} MB")

# Set path — Colab uploads files to /content/ by default
INPUT_PATH  = f"/content/{filename}"
OUTPUT_PATH = "/content/trust_paradox_clean.csv"

print(f"\n  Input  : {INPUT_PATH}")
print(f"  Output : {OUTPUT_PATH}  (will be created after cleaning)")

A file chooser will appear below.
Select: survey_results_public.csv (the raw Stack Overflow dataset)
File size is ~135 MB — upload may take 1-2 minutes on slower connections.



Saving survey_results_public.csv to survey_results_public.csv

✓ Uploaded: survey_results_public.csv
✓ Size: 134.4 MB

  Input  : /content/survey_results_public.csv
  Output : /content/trust_paradox_clean.csv  (will be created after cleaning)


In [3]:
# CELL 3 — STEP 1: Load and verify the raw dataset
# ════════════════════════════════════════════════════════════════════════════

print("Loading CSV... (may take 30-60 seconds for 135 MB file)")

df = pd.read_csv(INPUT_PATH, low_memory=False)

print(f"\n  Rows loaded    : {len(df):,}")
print(f"  Columns loaded : {len(df.columns)}")

# Hard checks — script stops here if the file is wrong
assert len(df) == 49191,    f"Expected 49,191 rows, got {len(df):,} — wrong file?"
assert len(df.columns) == 172, f"Expected 172 columns, got {len(df.columns)} — wrong file?"

print("\n✓ Dataset verified: 49,191 rows × 172 columns")
print("  Run Cell 4 — Duplicate check")

Loading CSV... (may take 30-60 seconds for 135 MB file)

  Rows loaded    : 49,191
  Columns loaded : 172

✓ Dataset verified: 49,191 rows × 172 columns
  Run Cell 4 — Duplicate check


In [5]:
 ## CELL 4 — STEP 2: Duplicate check


n_dupes = df.duplicated(subset=['ResponseId']).sum()
print(f"  Duplicate rows found : {n_dupes}")

if n_dupes > 0:
    df = df.drop_duplicates(subset=['ResponseId'])
    print(f"  Removed {n_dupes} duplicate rows.")
else:
    print("✓ Zero duplicates — dataset is clean at source")

  Duplicate rows found : 0
✓ Zero duplicates — dataset is clean at source


In [6]:
# CELL 5 — STEP 3: Missing value profile (look before touching)
# ════════════════════════════════════════════════════════════════════════════

KEY_COLS = [
    'AIAcc', 'AISent', 'AIThreat', 'AISelect', 'AIComplex',
    'AIFrustration', 'JobSat', 'YearsCode', 'DevType', 'Country',
    'Age', 'WorkExp', 'ToolCountWork',
]

print(f"\n  {'Column':<26} {'Non-Null':>9} {'Missing':>9} {'Missing %':>10}  Note")
print(f"  {'─'*72}")

for col in KEY_COLS:
    nn  = int(df[col].notna().sum())
    mis = len(df) - nn
    pct = mis / len(df) * 100
    note = "← MNAR — RETAIN, never delete" if pct > 30 else \
           "← cap outliers" if col == 'YearsCode' else \
           "← complete ✓"  if pct == 0 else ""
    print(f"  {col:<26} {nn:>9,} {mis:>9,} {pct:>9.1f}%  {note}")

print("""
─────────────────────────────────────────────────────────────────
  KEY INSIGHT: AIAcc (32.3%), AISent (32.0%), AISelect (31.5%)
  are ALL missing at almost the same rate. The same ~15,500
  developers are skipping all three AI questions. This is MNAR
  (Missing Not At Random). Deleting these rows would remove the
  most sceptical voices and inflate the trust rate artificially.
  → Decision: RETAIN all rows. Label blanks as "No Response".
─────────────────────────────────────────────────────────────────
""")


  Column                      Non-Null   Missing  Missing %  Note
  ────────────────────────────────────────────────────────────────────────
  AIAcc                         33,297    15,894      32.3%  ← MNAR — RETAIN, never delete
  AISent                        33,467    15,724      32.0%  ← MNAR — RETAIN, never delete
  AIThreat                      36,078    13,113      26.7%  
  AISelect                      33,720    15,471      31.5%  ← MNAR — RETAIN, never delete
  AIComplex                     33,283    15,908      32.3%  ← MNAR — RETAIN, never delete
  AIFrustration                 31,522    17,669      35.9%  ← MNAR — RETAIN, never delete
  JobSat                        26,670    22,521      45.8%  ← MNAR — RETAIN, never delete
  YearsCode                     43,042     6,149      12.5%  ← cap outliers
  DevType                       43,680     5,511      11.2%  
  Country                       35,437    13,754      28.0%  
  Age                           49,191         0  

In [7]:
## CELL 6 — STEP 4: Developer scope filter  (49,191 → 48,195)


KEEP_MAINBRANCH = [
    "I am a developer by profession",
    "I am not primarily a developer, but I write code sometimes as part of my work/studies",
    "I am learning to code",
    "I code primarily as a hobby",
    "I used to be a developer by profession, but no longer am",
]

rows_before = len(df)
df = df[df['MainBranch'].isin(KEEP_MAINBRANCH)].copy()
rows_after  = len(df)
removed     = rows_before - rows_after

print(f"  Rows before filter : {rows_before:,}")
print(f"  Rows after filter  : {rows_after:,}")
print(f"  Rows removed       : {removed}  (non-developer support roles only)")

assert removed == 996,        f"Expected 996 removed, got {removed}"
assert rows_after == 48195,   f"Expected 48,195 rows, got {rows_after:,}"

print("\n  Kept groups:")
for val, cnt in df['MainBranch'].value_counts().items():
    print(f"    {cnt:>7,}  {val[:70]}")

print(f"\n✓ Filter done: {rows_after:,} developer rows retained")


  Rows before filter : 49,191
  Rows after filter  : 48,195
  Rows removed       : 996  (non-developer support roles only)

  Kept groups:
     37,467  I am a developer by profession
      4,894  I am not primarily a developer, but I write code sometimes as part of 
      2,585  I am learning to code
      1,924  I code primarily as a hobby
      1,325  I used to be a developer by profession, but no longer am

✓ Filter done: 48,195 developer rows retained


In [8]:
## CELL 7 — STEP 5: Label blank values as "No Response"

# Original columns are NEVER modified. We create new _clean columns.

LABEL_COLS = {
    'AIAcc'    : 'AIAcc_clean',
    'AISent'   : 'AISent_clean',
    'AIThreat' : 'AIThreat_clean',
    'JobSat'   : 'JobSat_clean',
}

for src, dst in LABEL_COLS.items():
    df[dst] = df[src].fillna('No Response')
    blanks  = df[dst].isna().sum()
    nr_cnt  = (df[dst] == 'No Response').sum()
    print(f"  {src:<12} → {dst:<18}  "
          f"{nr_cnt:>6,} labelled 'No Response'  |  {blanks} blanks remaining")
    assert blanks == 0, f"ERROR: {dst} still has blank cells!"

print("""
─────────────────────────────────────────────────────────────────
  CORRECT percentage formula (use this always):

    Highly trust % = df[df['AIAcc_clean']=='Highly trust'].shape[0]
                   / df[df['AIAcc_clean']!='No Response'].shape[0]
                   = 1,000 / 32,717 = 3.1%  ✓

  WRONG formula (DO NOT use):
    = df[df['AIAcc_clean']=='Highly trust'].shape[0] / len(df)
    = 1,000 / 48,195 = 2.1%  ✗  (wrong denominator)
─────────────────────────────────────────────────────────────────
""")
print("✓ All clean columns created with zero blanks")

  AIAcc        → AIAcc_clean         15,478 labelled 'No Response'  |  0 blanks remaining
  AISent       → AISent_clean        15,311 labelled 'No Response'  |  0 blanks remaining
  AIThreat     → AIThreat_clean      12,793 labelled 'No Response'  |  0 blanks remaining
  JobSat       → JobSat_clean        21,525 labelled 'No Response'  |  0 blanks remaining

─────────────────────────────────────────────────────────────────
  CORRECT percentage formula (use this always):

    Highly trust % = df[df['AIAcc_clean']=='Highly trust'].shape[0]
                   / df[df['AIAcc_clean']!='No Response'].shape[0]
                   = 1,000 / 32,717 = 3.1%  ✓

  WRONG formula (DO NOT use):
    = df[df['AIAcc_clean']=='Highly trust'].shape[0] / len(df)
    = 1,000 / 48,195 = 2.1%  ✗  (wrong denominator)
─────────────────────────────────────────────────────────────────

✓ All clean columns created with zero blanks


In [9]:
## CELL 8 — STEP 6: Ordinal encoding  (AIAcc_score and AISent_score)


# Trust scale: 5 = Highly trust (max), 1 = Highly distrust (min)
TRUST_MAP = {
    'Highly trust'              : 5,
    'Somewhat trust'            : 4,
    'Neither trust nor distrust': 3,
    'Somewhat distrust'         : 2,
    'Highly distrust'           : 1,
    'No Response'               : np.nan,   # ← NaN NOT zero (excluded from mean)
}

# Sentiment scale: 5 = Very favorable, 1 = Very unfavorable
SENT_MAP = {
    'Very favorable'  : 5,
    'Favorable'       : 4,
    'Indifferent'     : 3,
    'Unfavorable'     : 2,
    'Very unfavorable': 1,
    'Unsure'          : np.nan,
    'No Response'     : np.nan,
}

df['AIAcc_score']  = df['AIAcc_clean'].map(TRUST_MAP)
df['AISent_score'] = df['AISent'].map(SENT_MAP)

# Verify
assert df.loc[df['AIAcc_clean']=='Highly trust',  'AIAcc_score'].iloc[0] == 5
assert df.loc[df['AIAcc_clean']=='Highly distrust','AIAcc_score'].iloc[0] == 1

mean_score   = df['AIAcc_score'].mean()
median_score = df['AIAcc_score'].median()
non_null     = df['AIAcc_score'].notna().sum()

print(f"  AIAcc_score mean     : {mean_score:.3f}  (below 3.0 = overall distrust direction)")
print(f"  AIAcc_score median   : {median_score:.1f}")
print(f"  AIAcc_score non-null : {non_null:,}")

print("\n  Trust distribution (n=32,717 who answered):")
trust_ans = df[df['AIAcc_clean'] != 'No Response']
for level in ['Highly trust','Somewhat trust','Neither trust nor distrust',
               'Somewhat distrust','Highly distrust']:
    cnt = (trust_ans['AIAcc_clean'] == level).sum()
    pct = cnt / len(trust_ans) * 100
    bar = '█' * int(pct / 2)
    print(f"    {level:<35} {cnt:>6,}  {pct:>5.1f}%  {bar}")

print("\n✓ Ordinal encoding complete")



  AIAcc_score mean     : 2.700  (below 3.0 = overall distrust direction)
  AIAcc_score median   : 3.0
  AIAcc_score non-null : 32,717

  Trust distribution (n=32,717 who answered):
    Highly trust                         1,000    3.1%  █
    Somewhat trust                       9,668   29.6%  ██████████████
    Neither trust nor distrust           7,039   21.5%  ██████████
    Somewhat distrust                    8,549   26.1%  █████████████
    Highly distrust                      6,461   19.7%  █████████

✓ Ordinal encoding complete


In [10]:
# CELL 9 — STEP 7: YearsCode — cap outliers + create ExperienceBand


# Convert to numeric first (handles any stray string values)
df['YearsCode_num'] = pd.to_numeric(df['YearsCode'], errors='coerce')

print(f"  YearsCode max BEFORE cap : {df['YearsCode_num'].max():.0f}  ← implausible")
print(f"  Values above 50          : {(df['YearsCode_num'] > 50).sum()}")

# Cap at 50 using np.minimum
# np.minimum(x, 50) → if x > 50, returns 50 ; if x ≤ 50, returns x ; NaN stays NaN
df['YearsCode_num'] = np.minimum(df['YearsCode_num'], 50)

print(f"  YearsCode max AFTER cap  : {df['YearsCode_num'].max():.0f}  ✓")

# Create 4-level ExperienceBand
def assign_band(years):
    if pd.isna(years):          return 'Unknown'
    elif years <= 2:            return 'Early Career (0-2 yrs)'
    elif years <= 5:            return 'Mid-Early (3-5 yrs)'
    elif years <= 10:           return 'Mid (6-10 yrs)'
    else:                       return 'Experienced (10+ yrs)'

df['ExperienceBand'] = df['YearsCode_num'].apply(assign_band)

print("\n  ExperienceBand distribution:")
for band, cnt in df['ExperienceBand'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f"    {band:<30} {cnt:>6,}  ({pct:.1f}%)")

assert df['ExperienceBand'].isna().sum() == 0, "ExperienceBand has blank cells!"
print("\n✓ YearsCode capped and ExperienceBand created")

  YearsCode max BEFORE cap : 100  ← implausible
  Values above 50          : 239
  YearsCode max AFTER cap  : 50  ✓

  ExperienceBand distribution:
    Experienced (10+ yrs)          25,523  (53.0%)
    Mid (6-10 yrs)                 10,192  (21.1%)
    Unknown                         5,905  (12.3%)
    Mid-Early (3-5 yrs)             4,993  (10.4%)
    Early Career (0-2 yrs)          1,582  (3.3%)

✓ YearsCode capped and ExperienceBand created


In [11]:
## CELL 10 — STEP 8: Create UsesAI binary adoption flag


USES_AI_VALUES = {
    "Yes, I use AI tools daily"                   : "Uses AI",
    "Yes, I use AI tools weekly"                  : "Uses AI",
    "Yes, I use AI tools monthly or infrequently" : "Uses AI",
    "No, but I plan to soon"                      : "Does Not Use AI",
    "No, and I don't plan to"                     : "Does Not Use AI",
}

def classify_ai_usage(val):
    if pd.isna(val):
        return 'No Response'
    return USES_AI_VALUES.get(val, 'Does Not Use AI')

df['UsesAI'] = df['AISelect'].apply(classify_ai_usage)

print("  UsesAI distribution:")
for cat, cnt in df['UsesAI'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f"    {cat:<22} {cnt:>6,}  ({pct:.1f}%)")

# Adoption rate
answered   = df[df['UsesAI'] != 'No Response']
adopt_rate = (answered['UsesAI'] == 'Uses AI').mean() * 100

print(f"\n  Adoption rate (among those who answered) : {adopt_rate:.1f}%")
print(f"  Stack Overflow published figure           : 84%")
print(f"  Difference                                : {84 - adopt_rate:.1f} pp")
print(f"  Reason for gap: SO includes 'plan to soon'; we count active users only")

assert abs(adopt_rate - 78.5) < 0.1, f"Adoption rate unexpected: {adopt_rate:.1f}%"
print("\n✓ UsesAI flag created — adoption rate confirmed at 78.5%")

  UsesAI distribution:
    Uses AI                26,013  (54.0%)
    No Response            15,063  (31.3%)
    Does Not Use AI         7,119  (14.8%)

  Adoption rate (among those who answered) : 78.5%
  Stack Overflow published figure           : 84%
  Difference                                : 5.5 pp
  Reason for gap: SO includes 'plan to soon'; we count active users only

✓ UsesAI flag created — adoption rate confirmed at 78.5%


In [12]:
# CELL 11 — STEP 9: ToolCountWork — document outliers + create ToolGroup


tc = pd.to_numeric(df['ToolCountWork'], errors='coerce')

print(f"  Non-null    : {tc.notna().sum():,}")
print(f"  Min         : {tc.min():.0f}")
print(f"  Median      : {tc.median():.1f}  ← correct measure (use this, not mean)")
print(f"  Mean        : {tc.mean():.1f}  ← distorted by outliers — do not use")
print(f"  Max         : {tc.max():.0f}  ← extreme outlier (data entry error)")
print(f"  Values > 100: {(tc > 100).sum()} rows")

def assign_tool_group(val):
    if pd.isna(val):   return 'No Response'
    elif val <= 1:     return '1 Tool'
    elif val <= 3:     return '2-3 Tools'
    else:              return '4+ Tools'

df['ToolGroup'] = tc.apply(assign_tool_group)

print("\n  ToolGroup distribution:")
for grp, cnt in df['ToolGroup'].value_counts().items():
    print(f"    {grp:<15} {cnt:>6,}")

print("\n✓ ToolGroup created for Paradox 2 analysis")

  Non-null    : 27,119
  Min         : 0
  Median      : 6.0  ← correct measure (use this, not mean)
  Mean        : 17.1  ← distorted by outliers — do not use
  Max         : 10000  ← extreme outlier (data entry error)
  Values > 100: 91 rows

  ToolGroup distribution:
    4+ Tools        21,810
    No Response     21,076
    2-3 Tools        4,358
    1 Tool             951

✓ ToolGroup created for Paradox 2 analysis


In [13]:
# CELL 12 — STEP 10: Multi-select column — count AIFrustration values
# ════════════════════════════════════════════════════════════════════════════
# AIFrustration cells contain semicolon-separated values like:
# "AI solutions that are almost right, but not quite;Debugging AI-generated..."
# str.contains(keyword) finds the keyword anywhere in the cell — handles this.

FRUSTRATION_KEYWORDS = {
    'Frust_AlmostRight'      : 'almost right',
    'Frust_Debugging'        : 'Debugging',
    'Frust_LessConfident'    : 'less confident',
    'Frust_HardToUnderstand' : 'hard to understand',
    'Frust_NoProblems'       : "haven't encountered",
}

frust_nonnull = df['AIFrustration'].notna().sum()
print(f"  AIFrustration non-null rows : {frust_nonnull:,}")
print(f"\n  {'Column':<28} {'Count':>8}  Note")
print(f"  {'─'*55}")

for col_name, keyword in FRUSTRATION_KEYWORDS.items():
    # str.contains: na=False treats NaN as False (not a match)
    mask  = df['AIFrustration'].str.contains(keyword, na=False, regex=False)
    count = int(mask.sum())
    df[col_name] = mask.astype(int)   # 1 = has this frustration, 0 = does not
    print(f"  {col_name:<28} {count:>8,}  (1=has it, 0=does not)")

print("\n✓ Frustration binary columns created")


  AIFrustration non-null rows : 30,985

  Column                          Count  Note
  ───────────────────────────────────────────────────────
  Frust_AlmostRight              20,465  (1=has it, 0=does not)
  Frust_Debugging                14,041  (1=has it, 0=does not)
  Frust_LessConfident             6,207  (1=has it, 0=does not)
  Frust_HardToUnderstand          5,050  (1=has it, 0=does not)
  Frust_NoProblems                    0  (1=has it, 0=does not)

✓ Frustration binary columns created


In [14]:
 ## CELL 13 — STEP 11: Print all paradox statistics


trust_ans = df[df['AIAcc_clean'] != 'No Response']
n_trust   = len(trust_ans)
adopt_rate = (df[df['UsesAI'] != 'No Response']['UsesAI'] == 'Uses AI').mean() * 100

ht_pct  = (trust_ans['AIAcc_clean'] == 'Highly trust').mean()  * 100
ad_pct  = trust_ans['AIAcc_clean'].isin(
              ['Somewhat distrust','Highly distrust']).mean() * 100

print("=" * 60)
print("  PARADOX 1 — USE-DESPITE-DISTRUST")
print("=" * 60)
print(f"  Adoption rate          : {adopt_rate:.1f}%")
print(f"  Highly trust %         : {ht_pct:.1f}%   (only 1 in 32 developers)")
print(f"  Any distrust %         : {ad_pct:.1f}%  (nearly half)")
print(f"  Paradox Severity Score : {adopt_rate - ht_pct:.1f} pts")

print(f"\n  Expertise effect:")
print(f"  {'Band':<30} {'High Trust %':>13} {'Any Distrust %':>15}")
for band in ['Early Career (0-2 yrs)','Mid-Early (3-5 yrs)',
              'Mid (6-10 yrs)','Experienced (10+ yrs)']:
    g = trust_ans[trust_ans['ExperienceBand'] == band]
    ht_g = (g['AIAcc_clean'] == 'Highly trust').mean() * 100
    ad_g = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    print(f"  {band:<30} {ht_g:>12.1f}% {ad_g:>14.1f}%")

print("\n" + "=" * 60)
print("  PARADOX 2 — ADOPTION-ETHICS GAP")
print("=" * 60)
print(f"  {'Tool Group':<15} {'Any Distrust %':>15}  n")
for grp in ['1 Tool', '2-3 Tools', '4+ Tools']:
    g = df[(df['ToolGroup'] == grp) & (df['AIAcc_clean'] != 'No Response')]
    ad = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    print(f"  {grp:<15} {ad:>14.1f}%  {len(g):,}")

print("\n" + "=" * 60)
print("  PARADOX 3 — JOB THREAT")
print("=" * 60)
threat_ans = df[df['AIThreat_clean'] != 'No Response']
for val in ['No', "I'm not sure", 'Yes']:
    cnt = (threat_ans['AIThreat_clean'] == val).sum()
    pct = cnt / len(threat_ans) * 100
    print(f"  {val:<20} {cnt:>6,}  ({pct:.1f}%)")


  PARADOX 1 — USE-DESPITE-DISTRUST
  Adoption rate          : 78.5%
  Highly trust %         : 3.1%   (only 1 in 32 developers)
  Any distrust %         : 45.9%  (nearly half)
  Paradox Severity Score : 75.5 pts

  Expertise effect:
  Band                            High Trust %  Any Distrust %
  Early Career (0-2 yrs)                 10.0%           22.8%
  Mid-Early (3-5 yrs)                     5.4%           34.9%
  Mid (6-10 yrs)                          2.8%           43.8%
  Experienced (10+ yrs)                   2.3%           49.8%

  PARADOX 2 — ADOPTION-ETHICS GAP
  Tool Group       Any Distrust %  n
  1 Tool                    41.4%  867
  2-3 Tools                 41.4%  4,121
  4+ Tools                  47.5%  20,951

  PARADOX 3 — JOB THREAT
  No                   22,576  (63.8%)
  I'm not sure          7,553  (21.3%)
  Yes                   5,273  (14.9%)


In [15]:
# CELL 14 — STEP 12: Final validation — all checks must say PASS


print("VALIDATION CHECKS\n")

checks = [
    ("Row count",                     len(df),                               48195),
    ("Zero blanks: AIAcc_clean",      df['AIAcc_clean'].isna().sum(),        0),
    ("Zero blanks: UsesAI",           df['UsesAI'].isna().sum(),             0),
    ("Zero blanks: ExperienceBand",   df['ExperienceBand'].isna().sum(),     0),
    ("Zero blanks: AIThreat_clean",   df['AIThreat_clean'].isna().sum(),     0),
    ("Highly trust count",            int((df['AIAcc_clean']=='Highly trust').sum()),  1000),
    ("Uses AI count",                 int((df['UsesAI']=='Uses AI').sum()),            26013),
    ("Experienced band count",        int((df['ExperienceBand']=='Experienced (10+ yrs)').sum()), 25523),
    ("No Response AIAcc count",       int((df['AIAcc_clean']=='No Response').sum()),  15478),
]

all_pass = True
for label, actual, expected in checks:
    ok = (actual == expected)
    status = "PASS ✓" if ok else f"FAIL ✗  (expected {expected:,})"
    print(f"  {label:<38} {str(actual):>8}  {status}")
    if not ok:
        all_pass = False

print()
if all_pass:
    print("  ✓  ALL 9 CHECKS PASSED — dataset is clean and verified")
else:
    print("  ✗  SOME CHECKS FAILED — review output above before saving")

VALIDATION CHECKS

  Row count                                 48195  PASS ✓
  Zero blanks: AIAcc_clean                      0  PASS ✓
  Zero blanks: UsesAI                           0  PASS ✓
  Zero blanks: ExperienceBand                   0  PASS ✓
  Zero blanks: AIThreat_clean                   0  PASS ✓
  Highly trust count                         1000  PASS ✓
  Uses AI count                             26013  PASS ✓
  Experienced band count                    25523  PASS ✓
  No Response AIAcc count                   15478  PASS ✓

  ✓  ALL 9 CHECKS PASSED — dataset is clean and verified


In [17]:
# CELL 15 — STEP 13: Save cleaned CSV + download it


# Save to Colab's /content/ folder
df.to_csv(OUTPUT_PATH, index=False)

file_mb = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
print(f"  Saved: {OUTPUT_PATH}")
print(f"  Rows  : {len(df):,}")
print(f"  Cols  : {len(df.columns)}  ({len(df.columns) - 172} new columns added)")
print(f"  Size  : {file_mb:.1f} MB")

print("\n  New columns added:")
new_cols = [
    'YearsCode_num','ExperienceBand',
    'AIAcc_clean','AIAcc_score',
    'AISent_clean','AISent_score',
    'AIThreat_clean','JobSat_clean',
    'UsesAI','ToolGroup',
    'Frust_AlmostRight','Frust_Debugging',
    'Frust_LessConfident','Frust_HardToUnderstand','Frust_NoProblems',
]
for c in new_cols:
    print(f"    {c}")

# ── Download the file to your computer ───────────────────────────────────────
print("\n  Downloading trust_paradox_clean.csv to your computer...")
from google.colab import files
files.download(OUTPUT_PATH)

print("✓ Cleaned data saved and download initiated.")

  Saved: /content/trust_paradox_clean.csv
  Rows  : 48,195
  Cols  : 187  (15 new columns added)
  Size  : 127.7 MB

  New columns added:
    YearsCode_num
    ExperienceBand
    AIAcc_clean
    AIAcc_score
    AISent_clean
    AISent_score
    AIThreat_clean
    JobSat_clean
    UsesAI
    ToolGroup
    Frust_AlmostRight
    Frust_Debugging
    Frust_LessConfident
    Frust_HardToUnderstand
    Frust_NoProblems



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Cleaned data saved and download initiated.
